## this notebook is used to run GWAS on the "Cohort Builder"-derived cohort

adapted from notebook:

"5 - CB GWAS MF"

in workspace "Hypothyroidism genomics v7"


In [ ]:
# !pip install polars

In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
import json
import re
from google.cloud import bigquery
import polars as pl
#for gwas
import logging
import matplotlib.pyplot as plt
import pyspark
import hail as hl
from tqdm.auto import tqdm
%matplotlib inline

In [ ]:
def wb(*args):
    """Run a wb command and return parsed JSON."""
    cmd = ["wb", *args, "--format=json"]
    result = subprocess.check_output(cmd, text=True)
    return json.loads(result)


# Get workspace info
workspace = wb("workspace", "describe")
google_project_id = workspace["googleProjectId"]

# Get resources
resources = wb("resource", "list")

# WORKSPACE_BUCKET
bucket_resources = [
    r for r in resources
    if r.get("resourceType") == "GCS_BUCKET"
    and "practical_considerations_bucket" in r.get("id", "")
    and "temporary" not in r.get("id", "")
]

if not bucket_resources:
    raise ValueError("No matching bucket found")

bucket = f"gs://{bucket_resources[0]['bucketName']}"

# WORKSPACE_CDR
bq_resources = [
    r for r in resources
    if r.get("resourceType") in {"BQ_DATASET", "BIGQUERY_DATASET"}
]

cdr_resources = [
    r for r in bq_resources
    if re.match(r"^C\d{4}Q\d+R\d+$", r.get("datasetId", ""))
]

if not cdr_resources:
    raise ValueError("No matching CDR dataset found")

CDR = (
    f"{cdr_resources[0]['projectId']}."
    f"{cdr_resources[0]['datasetId']}"
)


In [ ]:
#initialize (only run once)
hl.init(
    gcs_requester_pays_configuration=google_project_id
)

In [ ]:
logging.basicConfig(level='INFO')

In [ ]:
#read in acaf hail mt
# update to v9
# srWGS_snpindel_bucket = 'gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel'
srWGS_snpindel_bucket = 'gs://vwb-aou-datasets-controlled/v7/wgs/snpindel'
wgs_path = f'{srWGS_snpindel_bucket}/acaf_threshold_v7.1/splitMT/hail.mt'
wgs = hl.read_matrix_table(wgs_path, _n_partitions = 10000)

In [ ]:
#load in ancestry file
ancestry_pred_path = f'{srWGS_snpindel_bucket}/aux/ancestry/ancestry_preds.tsv'
ancestry_pred = hl.import_table(ancestry_pred_path,
                               key="research_id", 
                               impute=True, 
                               types={"research_id":"tstr","pca_features":hl.tarray(hl.tfloat)})

In [ ]:
# Define paths/filenames
pheno_path = '/home/dataproc/workspace/practical_considerations_bucket/hypothyroid_data/huan_phenotype_v4_covars.csv'
names = ['transancestry', 'eur', 'noneur', 'afr', 'amr']
sex = ['a','f','m']

pheno_path2 = '/home/dataproc/workspace/practical_considerations_bucket/hypothyroid_data/cb_v2_phenotype_covars.tsv'

In [ ]:
# Annotate wgs with ancestry
wgs = wgs.annotate_cols(ancestry_pred = ancestry_pred[wgs.s])